In [ ]:
from pathlib import Path
from typing import List

import numpy as np
import torch
import pytorch_lightning as pl
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DataModule
# ─────────────────────────────────────────────────────────────────────────────

class GridDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = ".", batch_size: int = 128):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.batch_size = batch_size
        self.train_ds: List[Data] | None = None
        self.val_ds: List[Data] | None = None
        self.test_ds: List[Data] | None = None

    def prepare_data(self):
        # nothing to download, but Trigger for DDP
        pass

    def setup(self, stage: str | None = None):
        self.train_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn/train.pt"), 
        self.val_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn/val.pt")
        self.test_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn/val.pt")

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size)
    
    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size)

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
import pytorch_lightning as pl
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

class GCNModule(pl.LightningModule):
    def __init__(self, in_feats: int = 5, hidden: int = 64, 
                 out_feats: int = 1, num_layers: int = 8, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()

        # Собираем слой за слоем
        layers = []
        # Первый слой: in_feats -> hidden
        layers.append(GCNConv(in_feats, hidden))
        # Промежуточные: hidden -> hidden
        for _ in range(num_layers - 2):
            layers.append(GCNConv(hidden, hidden))
        # Последний слой: hidden -> out_feats
        layers.append(GCNConv(hidden, out_feats))
        self.convs = nn.ModuleList(layers)

        self.loss_fn = nn.MSELoss()

    def forward(self, data: Data):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_weight
        # Прогоняем через все conv-слои
        for conv in self.convs[:-1]:
            x = conv(x, edge_index, edge_weight)
            x = F.relu(x)
        # Последний слой без активации (и убираем размерность)
        x = self.convs[-1](x, edge_index, edge_weight).squeeze(-1)
        return x

    def _step(self, batch: Data, stage: str):
        preds = self(batch)
        loss = self.loss_fn(preds, batch.y)
        self.log(f"{stage}_mse", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch: Data, batch_idx: int):
        return self._step(batch, "train")

    def validation_step(self, batch: Data, batch_idx: int):
        self._step(batch, "val")

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


In [ ]:
dm = GridDataModule(data_dir="/home/silvarum/TransPath_Adaptation/gcn_dataset", batch_size=128)

In [13]:
model = GCNModule(hidden=64, lr=1e-2)

logger = WandbLogger(project="gcn-cf", name="ok data preparation_4", resume="never", reinit=True)

ckpt_cb = ModelCheckpoint(
    monitor="val_mse",          # метрика, за которой следим
    mode="min",                 # «меньше — лучше»
    dirpath="checkpoints/",     # куда класть файлы
    filename="best-{epoch}-{val_mse:.4f}",
    save_top_k=3,               # хранить только лучший
)
trainer = pl.Trainer(
    max_epochs=10,
    logger=logger,
    accelerator="cuda",
    devices=[6],
    callbacks=[ckpt_cb],        # ← добавили
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [14]:
trainer.fit(model, datamodule=dm)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: Currently logged in as: alex26-std (alex26-std-saint-petersburg-state-university). Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


/tmp/ipykernel_2053191/1471452430.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.train_ds, self.val_ds = torch.load("/home/silvarum/TransPath_Adaptation/gcn/train

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/silvarum/miniconda3/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=255` in the `DataLoader` to improve performance.
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 524288. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/home/silvarum/miniconda3/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [ ]:
# после trainer.fit(...)
# 1) загружаем лучшую модель из чекпоинта
ckpt = ModelCheckpoint(best_model_path="/home/silvarum/TransPath_Adaptation/checkpoints/best-epoch=53-val_mse=0.0295.ckpt")
best_ckpt = ckpt.best_model_path
model = GCNModule.load_from_checkpoint(best_ckpt)
model.eval()

# 2) создаём DataModule и подготавливаем (чтобы инициализировать даталоадеры)
dm = GridDataModule(data_dir="/home/silvarum/TransPath_Adaptation/gcn_dataset", batch_size=128)
dm.setup()

# 3) прогоняем predict по train/val/test (или любому из них)
#    predict вернёт список батчей, каждый — тензор [batch_size, num_nodes]
test_preds   = trainer.predict(model, datamodule=dm, dataloader=dm.test_dataloader())
# если есть тест:
# test_preds  = trainer.predict(model, datamodule=dm, dataloader=dm.test_dataloader())

# train_preds — список тензоров; склеим в один [N_total, num_nodes]
train_preds = torch.cat(test_preds, dim=0)  

# 4) сохраняем или анализируем
torch.save(train_preds, "predicted_heuristics_train.pt")
